In [3]:
import numpy as np
import random
import matplotlib.pyplot as plt
import h5py
from typing import List


### Structure d'un groupe

```python
group: {
  values: [0, 1, 2, 0, 0, 0, 4, 1, 1],
  cases: [
    {
      id: 1,
      value: 6
    },
    {
      id: 14,
      value: 2
    }
  ]
}
```

In [33]:
class Case:
  def __init__(self, id:int, value):
    self.id = id
    self.value = value

class Group: # TODO - Attention définition du calcul de l'énergie = règles transgréssées
  def __init__(self, cases: List[Case], n:int):
    # On stocke maintenant les cases sous forme de dictionnaire {id: value}
    self.cases = {case.id: case.value for case in cases}

    values = [0] * n
    for case in cases:
      values[case.value - 1] += 1

    self.values = values

  def find_case_value(self, id:int):
    return self.cases[id]
  
  def update(self, id:int, new_value:int):
    # Update values
    previous_value = self.cases[id]
    self.values[previous_value - 1] -= 1
    self.values[new_value - 1] += 1

    # Update case
    self.cases[id] = new_value
      
  def calculate_group_delta_energy(self, id:int, new_value:int) -> int:
    delta_energy = 0
    previous_value = self.cases[id]
    values = self.values.copy()

    if (previous_value == new_value):
      return 0
    
    values[previous_value - 1] -= 1
    if (values[previous_value - 1] == 1):
      delta_energy -= 1
    else:
      delta_energy += 1

    values[new_value - 1] += 1
    if (values[new_value - 1] == 1):
      delta_energy -= 1
    else:
      delta_energy += 1

    return delta_energy

class Sudoku: # TODO - attention pour groups_to_sudoku_structure ici je considère sudoku '3D conditions' = 2D
  def __init__(self, size):
    self.size = size

    # Init Groups
    def init_groups() -> List[Group]:
      cases = []
      groups = []

      for i in range(size * size):
        case_value = np.random.randint(1, size+1)
        cases.append(Case(i, case_value))

      # Groups creation
      for i in range(size):
        # Line creation
        line = cases[i * size : (i + 1) * size]
        
        # Column creation
        column = cases[i::size]

        # Bloc creation
        n = int(np.sqrt(size))
        r, c = divmod(i, n)
        block = []
        for rr in range(n):
            for cc in range(n):
                block.append(cases[(r * n + rr) * size + (c * n + cc)])

        # Append the Group to groups
        for group in [Group(line, size), Group(column, size), Group(block, size)]:
          groups.append(group)

      return groups
      
    self.groups = init_groups()

    # Create the dictionnary of indexes
    self.dict = {i: [] for i in range(size * size)}
    for i, group in enumerate(self.groups):
      # group.cases est maintenant un dictionnaire, on itère sur les clés (ids)
      for case_id in group.cases:
        self.dict[case_id].append(i)
  
    # Calculate the sudoku init energy
    self.energy = 0
    for group in self.groups:
        for val in group.values:
            if val != 1:
                self.energy += 1

  def update(self, delta_energy, id, new_value):
    # Update the sudoku energy
    self.energy += delta_energy

    # Update the case value and values of group
    for i in self.dict[id]:
      group = self.groups[i]
      group.update(id, new_value)

  def calculate_delta_energy(self, id, new_value):
    delta_energy = 0

    for i in self.dict[id]:
      group = self.groups[i]
      delta_energy += group.calculate_group_delta_energy(id, new_value)
    
    return delta_energy

  def groups_to_sudoku_shape(self):
    sudoku_structure = []
    for i in range(self.size**2):
      group_id = self.dict[i][0]
      val = self.groups[group_id].find_case_value(i)
      sudoku_structure.append(val)

    return np.array(sudoku_structure).reshape(self.size, self.size)

  def show(self):
    N = self.size
    n = int(np.sqrt(N))
    sudoku_shape = self.groups_to_sudoku_shape()

    print(f'Sudoku of size {N}x{N}, current energy: {self.energy}\n')
    
    cell_width = len(str(np.max(sudoku_shape))) + 2

    for i in range(n * n):
      # Horizontal block separator
      if i % n == 0 and i != 0:
          print("-" * ((n * n) * cell_width + (n - 1) * 3))

      row_str = []
      for j in range(n * n):

        # Vertical block separator
        if j % n == 0 and j != 0:
            row_str.append(" | ")

        # Center each number
        row_str.append(f"{sudoku_shape[i, j]:^{cell_width}}")

      print("".join(row_str))


class Metropolis:
  def __init__(self, n=None, sudoku=None):
    if sudoku is not None:
      self.sudoku = sudoku
    elif n is not None:
      self.sudoku = Sudoku(n)
    else:
      raise ValueError("Either 'n' or 'sudoku' must be provided.")

  def simulate(self, T, nb_iter):
    sudoku = self.sudoku
    N = sudoku.size

    # Track energy evolution
    Es = [sudoku.energy]

    for _ in range(nb_iter):
      spin_id = np.random.randint(0, N**2)
      new_value = np.random.randint(1, N + 1)

      dE = sudoku.calculate_delta_energy(spin_id, new_value)
      p = np.exp(- dE / T)

      if dE <= 0 or np.random.rand() <= p:
        sudoku.update(dE, spin_id, new_value)

      Es.append(sudoku.energy)

    self.Es = Es

  def see_evolution(self):
    plt.plot(np.arange(len(self.Es)), self.Es)

    plt.xlabel('Nombre itération Metropolis')
    plt.ylabel('Energie du sudoku')
    plt.title(f'Evolution de l\'énergie d\'une grille de sudoku {self.sudoku.size}x{self.sudoku.size}')
    plt.show()

In [45]:
m9 = Metropolis(625)

In [ ]:
m9.simulate(0.4, 2000000)
m9.see_evolution()

In [ ]:
de